# Pre Processing & Feature Extraction

### Imports

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from skimage.feature import hog, local_binary_pattern
from scipy.stats import kurtosis, skew
from sklearn.ensemble import RandomForestClassifier

### Definitions

In [ ]:
# Function to display sample images
def show_sample_images(folder, num_samples=5):
    sample_files = os.listdir(folder)[:num_samples]
    fig, axes = plt.subplots(1, num_samples, figsize=(15, 5))
    
    for i, file_name in enumerate(sample_files):
        img = cv2.imread(os.path.join(folder, file_name), cv2.IMREAD_GRAYSCALE)
        axes[i].imshow(img, cmap='gray')
        axes[i].set_title(file_name)
        axes[i].axis('off')
    
    plt.show()

In [ ]:
# Image processing function
def process_image(image_path, output_path, size=(256, 256)):
    try:
        # Read image
        image = cv2.imread(image_path)

        # Convert to grayscale
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # Apply Gaussian Blur for noise reduction
        blurred = cv2.GaussianBlur(gray, (3, 3), 0)

        # Resize to 256x256
        resized = cv2.resize(blurred, size, interpolation=cv2.INTER_AREA)

        # Save processed image
        cv2.imwrite(output_path, resized)
    except Exception as e:
        print(f"Error processing {image_path}: {e}")

In [ ]:
def preprocess_image(image_path, output_path, size=(256, 128)):
    """Preprocess an image: Convert to grayscale, denoise, resize, and apply binarization."""
    
    # Read image
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    # Check if the image was loaded properly
    if img is None:
        print(f"Skipping {image_path}: Unable to read.")
        return

    # Apply Gaussian Blur for noise reduction
    img = cv2.GaussianBlur(img, (3, 3), 0)

    # Resize image
    img = cv2.resize(img, size, interpolation=cv2.INTER_AREA)

    # Apply adaptive thresholding (better for signature images)
    img = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                cv2.THRESH_BINARY_INV, 11, 2)

    # Save processed image
    cv2.imwrite(output_path, img)

In [ ]:
def process_dataset(source_folder, target_folder):
    """Processes all images in a folder and saves them to the target folder."""
    
    # Ensure target directory exists
    os.makedirs(target_folder, exist_ok=True)

    # Process images with tqdm progress bar
    for file_name in tqdm(os.listdir(source_folder), desc=f"Processing {source_folder}"):
        input_path = os.path.join(source_folder, file_name)
        output_path = os.path.join(target_folder, file_name)
        preprocess_image(input_path, output_path)

In [ ]:
# Function to apply augmentations
def augment_image(image, augment_type):
    if augment_type == "rotate":
        # Rotate the image by a random angle
        angle = np.random.uniform(-30, 30)  # Random angle between -30 and 30 degrees
        h, w = image.shape[:2]
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        augmented_image = cv2.warpAffine(image, M, (w, h))

    elif augment_type == "scale":
        # Scale the image by a random factor
        scale = np.random.uniform(0.8, 1.2)  # Random scale factor between 0.8 and 1.2
        h, w = image.shape[:2]
        augmented_image = cv2.resize(image, (int(w * scale), int(h * scale)))

    elif augment_type == "shear":
        # Shear the image
        shear_factor = np.random.uniform(-0.2, 0.2)  # Random shear factor
        M = np.array([[1, shear_factor, 0], [0, 1, 0]])
        augmented_image = cv2.warpAffine(image, M, (image.shape[1], image.shape[0]))

    elif augment_type == "translate":
        # Translate the image by a random amount
        tx = np.random.randint(-20, 20)  # Random translation in x
        ty = np.random.randint(-20, 20)  # Random translation in y
        M = np.float32([[1, 0, tx], [0, 1, ty]])
        augmented_image = cv2.warpAffine(image, M, (image.shape[1], image.shape[0]))

    elif augment_type == "gaussian_noise":
        # Add Gaussian noise to the image
        noise = np.random.normal(0, 25, image.shape).astype(np.uint8)  # Mean=0, Std=25
        augmented_image = cv2.add(image, noise)

    return augmented_image

In [ ]:
# Function to perform augmentations on all images in a directory
def augment_images(input_dir, output_dir):
    for file_name in tqdm(os.listdir(input_dir), desc=f"Augmenting images in {input_dir}"):
        input_path = os.path.join(input_dir, file_name)
        image = cv2.imread(input_path)

        # Generate 3 augmentations for each image
        for i in range(3):
            augment_type = np.random.choice(["rotate", "scale", "shear", "translate", "gaussian_noise"])
            augmented_image = augment_image(image, augment_type)

            # Save the augmented image
            output_path = os.path.join(output_dir, f"{os.path.splitext(file_name)[0]}_aug_{i}.jpg")
            cv2.imwrite(output_path, augmented_image)

### Paths & Directory Setup

In [ ]:
# Paths
datasetDir_path = "../Dataset/Processing/"
target_genuine = os.path.join(datasetDir_path, "HybridDataset/Genuine")
target_forged = os.path.join(datasetDir_path, "HybridDataset/Forged")

# Output directories for processed images
processed_genuine = os.path.join(datasetDir_path, "Pre-ProcessedDataset/Genuine")
processed_forged = os.path.join(datasetDir_path, "Pre-ProcessedDataset/Forged")

# Output directories for Augmentationed images
augmented_genuine = os.path.join(datasetDir_path, "AugmentedDataset/Genuine")
augmented_forged = os.path.join(datasetDir_path, "AugmentedDataset/Forged")

# Ensure output directories exist
os.makedirs(os.path.join(datasetDir_path, "Pre-ProcessedDataset"), exist_ok=True)
os.makedirs(processed_genuine, exist_ok=True)
os.makedirs(processed_forged, exist_ok=True)

os.makedirs(os.path.join(datasetDir_path, "AugmentedDataset"), exist_ok=True)
os.makedirs(augmented_genuine, exist_ok=True)
os.makedirs(augmented_forged, exist_ok=True)

print("Paths & Directory setup done")

For Diverse Dataset Paths

In [ ]:
target_genuine = "../Dataset/Processing/DiverseDataset/HybridDatasets/Genuine"
target_forged = "../Dataset/Processing/DiverseDataset/HybridDatasets/Forged"

In [ ]:
print("Displaying sample images before processing...")
show_sample_images(target_genuine)
show_sample_images(target_forged)

### Pre-Processing

In [ ]:
# Process genuine images
print("Processing Genuine Signatures...")
for file_name in tqdm(os.listdir(target_genuine)):
    input_path = os.path.join(target_genuine, file_name)
    output_path = os.path.join(processed_genuine, file_name)
    process_image(input_path, output_path)

# Process forged images
print("Processing Forged Signatures...")
for file_name in tqdm(os.listdir(target_forged)):
    input_path = os.path.join(target_forged, file_name)
    output_path = os.path.join(processed_forged, file_name)
    process_image(input_path, output_path)

print("Preprocessing Complete! Processed images are saved in 'ProcessedDataset'.")


In [ ]:
print("Displaying sample images after preprocessing...")
show_sample_images(processed_genuine)
show_sample_images(processed_forged)

### Augmentations

In [ ]:
# Process augmentations for both genuine and forged images
augment_images(processed_genuine, augmented_genuine)
augment_images(processed_forged, augmented_forged)

print("Image augmentation complete!")

## Feature Extraction

### Definitions

In [ ]:
def extract_hog_features(img):
    """Extracts Histogram of Oriented Gradients (HOG) features."""
    features, _ = hog(img, pixels_per_cell=(16, 16), cells_per_block=(2, 2), 
                      orientations=9, visualize=True, block_norm='L2-Hys')
    return features

In [ ]:
def extract_lbp_features(img):
    """Extracts Local Binary Pattern (LBP) features."""
    lbp = local_binary_pattern(img, P=24, R=3, method="uniform")
    (hist, _) = np.histogram(lbp.ravel(), bins=np.arange(0, 27), range=(0, 26))
    hist = hist.astype("float")
    hist /= hist.sum()  # Normalize histogram
    return hist

In [ ]:
def extract_hu_moments(img):
    """Extracts shape-based Hu Moments features."""
    moments = cv2.moments(img)
    hu_moments = cv2.HuMoments(moments).flatten()
    hu_moments = -np.sign(hu_moments) * np.log10(np.abs(hu_moments))  # Log scale
    return hu_moments

In [ ]:
def extract_orb_features(img):
    """Extracts ORB keypoints and descriptors."""
    orb = cv2.ORB_create()
    keypoints, descriptors = orb.detectAndCompute(img, None)
    return len(keypoints)  # Number of keypoints detected

In [ ]:
def extract_pixel_intensity(img):
    """Extracts basic pixel intensity statistics."""
    return [np.mean(img), np.std(img), np.median(img), skew(img.flatten()), kurtosis(img.flatten())]

In [ ]:
def extract_pressure_features(img):
    """Estimates pressure points based on stroke thickness & pixel density."""
    binary = cv2.threshold(img, 128, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    thickness = np.sum(binary == 255) / np.count_nonzero(binary)  # Ratio of ink pixels
    density = np.sum(binary == 255) / binary.size  # Fraction of inked area
    return [thickness, density]

In [ ]:
def extract_features(image_path):
    """Extracts all features from a given image."""
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    
    img = cv2.resize(img, (256, 128))  # Resize to fixed size
    
    features = []
    features.extend(extract_hog_features(img))
    features.extend(extract_lbp_features(img))
    features.extend(extract_hu_moments(img))
    features.append(extract_orb_features(img))
    features.extend(extract_pixel_intensity(img))
    features.extend(extract_pressure_features(img))
    
    return features

In [ ]:
def process_dataset(input_folder, label):
    """Extracts features from all images in a dataset directory."""
    data = []
    for file_name in tqdm(os.listdir(input_folder), desc=f"Extracting Features from {label}"):
        image_path = os.path.join(input_folder, file_name)
        features = extract_features(image_path)
        if features:
            data.append([file_name, label] + features)
    return data

For Visualisations

In [ ]:
def show_sample_images(folder, num_samples=5):
    """Displays a few sample images from the dataset."""
    sample_files = os.listdir(folder)[:num_samples]
    fig, axes = plt.subplots(1, num_samples, figsize=(15, 5))
    
    for i, file_name in enumerate(sample_files):
        img = cv2.imread(os.path.join(folder, file_name), cv2.IMREAD_GRAYSCALE)
        axes[i].imshow(img, cmap='gray')
        axes[i].set_title(file_name)
        axes[i].axis('off')
    
    plt.show()

In [ ]:
def plot_feature_distribution(df):
    """Plots distribution of the extracted features."""
    plt.figure(figsize=(12, 6))
    for i in range(1, 6):  # Show first 5 features
        sns.kdeplot(df[f"feature_{i}"], label=f"Feature {i}", fill=True)
    
    plt.xlabel("Feature Value")
    plt.ylabel("Density")
    plt.title("Feature Distribution")
    plt.legend()
    plt.show()

In [ ]:
def plot_feature_importance(X, y):
    """Trains a simple classifier and plots feature importance."""
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X, y)
    importance = model.feature_importances_

    plt.figure(figsize=(12, 5))
    sns.barplot(x=range(len(importance)), y=importance)
    plt.xlabel("Feature Index")
    plt.ylabel("Importance Score")
    plt.title("Feature Importance for Signature Forgery Detection")
    plt.show()

### Feature Extraction

In [ ]:
# Paths to augmented images
genuine_path = "../Dataset/Processing/AugmentedDataset/Genuine"
forged_path = "../Dataset/Processing/AugmentedDataset/Forged"

# Show sample images from both datasets
show_sample_images(genuine_path)
show_sample_images(forged_path)

In [ ]:
# Extract features for both classes
genuine_data = process_dataset(genuine_path, 0)  # Label 0 for genuine
forged_data = process_dataset(forged_path, 1)    # Label 1 for forged

# Combine and save as CSV
columns = ["filename", "label"] + [f"feature_{i}" for i in range(len(genuine_data[0]) - 2)]
df = pd.DataFrame(genuine_data + forged_data, columns=columns)

# Plot feature distribution
plot_feature_distribution(df)

In [ ]:
# Saving Extracted Features
df.to_csv("../Dataset/Features/signature_features.csv", index=False)

print("Feature extraction completed and saved!")

Optional

In [ ]:
# Extract features and labels
X = df.iloc[:, 2:].values  # Feature columns
y = df["label"].values

# Plot feature importance
plot_feature_importance(X, y)